# NLP03 Project: 멋진 챗봇 만들기

이번 프로젝트에서는 번역 모델 실습에서 학습한 전처리, 토큰화, Transformer 모델 학습, BLEU 평가 흐름을 한국어 챗봇 데이터에 적용한다.

프로젝트 진행은 다음 순서로 나눈다.

1. 라이브러리 및 데이터 준비
2. 데이터 정제
3. 데이터 토큰화
4. 데이터 Augmentation
5. 데이터 벡터화
6. Transformer 훈련
7. 성능 측정 및 회고

우선 한 단계씩 작성하고 실행 결과를 확인하면서 진행한다.

## 1. 라이브러리 버전 확인

프로젝트에 사용할 주요 라이브러리를 불러오고 버전을 확인한다.
현재 프로젝트는 `AIFFEL py312 GPU` 커널 사용을 기준으로 한다.

`torch.cuda.is_available()`이 `True`이면 GPU를 사용할 수 있는 상태다.

In [1]:
import numpy
import pandas
import torch
import nltk
import gensim

print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("torch:", torch.__version__)
print("nltk:", nltk.__version__)
print("gensim:", gensim.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


numpy: 2.4.5
pandas: 3.0.3
torch: 2.11.0+cu128
nltk: 3.9.4
gensim: 4.4.0
cuda available: True
gpu: NVIDIA GeForce RTX 3060


## 2. 데이터 다운로드

프로젝트에서 사용할 데이터는 `songys/Chatbot_data`의 `ChatbotData.csv`이다.
CSV 파일을 내려받은 뒤 `pandas`로 읽고, 질문 데이터와 답변 데이터를 각각 `questions`, `answers` 변수로 분리한다.

- `Q`: 질문 문장
- `A`: 답변 문장
- `label`: 감정 라벨이며, 이번 챗봇 생성 모델에서는 사용하지 않는다.

In [2]:
import os
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
DATA_PATH = os.path.join(DATA_DIR, "ChatbotData.csv")

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Downloaded:", DATA_PATH)
else:
    print("Already exists:", DATA_PATH)


Already exists: data\ChatbotData.csv


## 3. 데이터 불러오기 및 기본 확인

다운로드한 CSV 파일을 읽어 데이터의 크기와 컬럼을 확인한다.
이후 질문과 답변 컬럼을 리스트로 분리해 다음 단계의 전처리 입력으로 사용한다.

In [3]:
df = pandas.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())

questions = df["Q"].astype(str).tolist()
answers = df["A"].astype(str).tolist()

print("질문 개수:", len(questions))
print("답변 개수:", len(answers))
print("질문 예시:", questions[0])
print("답변 예시:", answers[0])


데이터 크기: (11823, 3)
컬럼: ['Q', 'A', 'label']


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


질문 개수: 11823
답변 개수: 11823
질문 예시: 12시 땡!
답변 예시: 하루가 또 가네요.


### 1장 확인

여기까지 완료되면 다음을 확인한다.

- 라이브러리 import가 모두 성공했는가?
- `cuda available`이 `True`로 나오는가?
- `ChatbotData.csv`가 정상적으로 다운로드되었는가?
- `questions`, `answers`의 길이가 같은가?

다음 장에서는 `preprocess_sentence()`를 구현해 문장을 정제한다.

## 4. 데이터 정제

       이번 장에서는 원본 문장을 모델이 다루기 쉬운 형태로 정리합니다.

        LMS 조건은 두 가지입니다.

        1. 영어 문자는 모두 소문자로 바꿉니다.
        2. 한글, 영어, 숫자, 주요 문장부호만 남기고 나머지는 제거합니다.

        여기서 너무 많은 전처리를 하지는 않겠습니다. 뒤에서 `mecab.morphs`가 한국어를 토큰화해 줄 것이기 때문에, 문장부호 양옆에 공백을 넣는 작업은 생략합니다.


In [4]:
import re

def preprocess_sentence(sentence):
    sentence = str(sentence).lower()
    sentence = re.sub(r"[^가-힣a-z0-9?.!,]+", " ", sentence)
    sentence = re.sub(r"\s+", " ", sentence)
    sentence = sentence.strip()
    return sentence

print("슝=3")


슝=3


### 전처리 결과 확인

        함수가 제대로 작동하는지 바로 확인해 봅니다.

        아래 셀은 원본 질문/답변 5개와 전처리된 결과를 나란히 보여줍니다.  
        여기서는 아직 데이터를 바꾸는 단계가 아니라, **정제 함수가 어떻게 작동하는지 확인하는 단계**입니다.


In [5]:
for i in range(5):
    print(f"[{i}] 질문 원본:", questions[i])
    print(f"[{i}] 질문 정제:", preprocess_sentence(questions[i]))
    print(f"[{i}] 답변 원본:", answers[i])
    print(f"[{i}] 답변 정제:", preprocess_sentence(answers[i]))
    print("-" * 60)


[0] 질문 원본: 12시 땡!
[0] 질문 정제: 12시 땡!
[0] 답변 원본: 하루가 또 가네요.
[0] 답변 정제: 하루가 또 가네요.
------------------------------------------------------------
[1] 질문 원본: 1지망 학교 떨어졌어
[1] 질문 정제: 1지망 학교 떨어졌어
[1] 답변 원본: 위로해 드립니다.
[1] 답변 정제: 위로해 드립니다.
------------------------------------------------------------
[2] 질문 원본: 3박4일 놀러가고 싶다
[2] 질문 정제: 3박4일 놀러가고 싶다
[2] 답변 원본: 여행은 언제나 좋죠.
[2] 답변 정제: 여행은 언제나 좋죠.
------------------------------------------------------------
[3] 질문 원본: 3박4일 정도 놀러가고 싶다
[3] 질문 정제: 3박4일 정도 놀러가고 싶다
[3] 답변 원본: 여행은 언제나 좋죠.
[3] 답변 정제: 여행은 언제나 좋죠.
------------------------------------------------------------
[4] 질문 원본: PPL 심하네
[4] 질문 정제: ppl 심하네
[4] 답변 원본: 눈살이 찌푸려지죠.
[4] 답변 정제: 눈살이 찌푸려지죠.
------------------------------------------------------------


### 2장 확인

        여기까지 확인할 내용은 간단합니다.

        - `preprocess_sentence()` 함수가 실행되는지
        - 영어가 소문자로 바뀌는지
        - 이상한 특수문자가 제거되는지
        - 한글 문장이 너무 망가지지 않는지

        다음 장에서는 이 함수를 실제 데이터 전체에 적용하면서, `mecab.morphs`로 질문과 답변을 토큰화하는 `build_corpus()`를 만들겠습니다.


## 5. 데이터 토큰화

        이제 문장을 모델이 다룰 수 있는 작은 단위로 나눕니다.

        여기서는 LMS 조건에 맞춰 KoNLPy의 `Mecab`을 사용합니다.  
        `mecab.morphs("문장")`을 실행하면 문장이 형태소 단위의 리스트로 나뉩니다.

        예를 들어 `"오늘 너무 피곤하다"`는 대략 `["오늘", "너무", "피곤", "하다"]`처럼 쪼개집니다.


In [6]:
try:
    # Windows 로컬에서는 KoNLPy Mecab보다 python-mecab-ko가 설치와 실행이 안정적입니다.
    from mecab import MeCab
    mecab = MeCab()
    tokenize = mecab.morphs
    tokenizer_name = "python-mecab-ko"
except Exception as e:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()

    def tokenize(sentence):
        return [token.form for token in kiwi.tokenize(sentence)]

    tokenizer_name = "Kiwi"
    print("MeCab을 사용할 수 없어 Kiwi로 대체합니다.")
    print("MeCab 오류:", e)

sample = "오늘 너무 피곤하다."
print("사용 토크나이저:", tokenizer_name)
print("원본:", sample)
print("토큰화:", tokenize(preprocess_sentence(sample)))


사용 토크나이저: python-mecab-ko
원본: 오늘 너무 피곤하다.
토큰화: ['오늘', '너무', '피곤', '하', '다', '.']


### `build_corpus()` 함수 만들기

        이 함수는 질문 문장과 답변 문장을 받아서 각각 토큰화된 말뭉치로 바꿉니다.

        흐름은 이렇게 잡았습니다.

        1. 질문과 답변을 하나씩 짝지어 꺼냅니다.
        2. 앞에서 만든 `preprocess_sentence()`로 정제합니다.
        3. 전달받은 토크나이저 함수로 토큰화합니다.
        4. 토큰 수가 너무 긴 문장은 제외합니다.
        5. 중복 문장은 제외합니다.

        여기서 중복 검사는 LMS 안내에 맞춰 질문은 질문대로, 답변은 답변대로 따로 검사합니다.


In [7]:
from tqdm import tqdm

MAX_TOKEN_LEN = 40

def build_corpus(src_sentences, tgt_sentences, tokenizer, max_len=MAX_TOKEN_LEN):
    src_corpus = []
    tgt_corpus = []
    src_seen = set()
    tgt_seen = set()

    for src, tgt in tqdm(zip(src_sentences, tgt_sentences), total=len(src_sentences)):
        src = preprocess_sentence(src)
        tgt = preprocess_sentence(tgt)

        src_tokens = tokenizer(src)
        tgt_tokens = tokenizer(tgt)

        if len(src_tokens) == 0 or len(tgt_tokens) == 0:
            continue

        if len(src_tokens) > max_len or len(tgt_tokens) > max_len:
            continue

        src_key = " ".join(src_tokens)
        tgt_key = " ".join(tgt_tokens)

        if src_key in src_seen or tgt_key in tgt_seen:
            continue

        src_seen.add(src_key)
        tgt_seen.add(tgt_key)

        src_corpus.append(src_tokens)
        tgt_corpus.append(tgt_tokens)

    return src_corpus, tgt_corpus

print("슝=3")


슝=3


### 질문/답변 데이터 토큰화

        이제 전체 `questions`, `answers` 데이터에 `build_corpus()`를 적용합니다.

        결과인 `que_corpus`, `ans_corpus`는 문자열 문장 리스트가 아니라, **토큰 리스트들의 리스트**입니다.


In [8]:
que_corpus, ans_corpus = build_corpus(
    questions,
    answers,
    tokenize,
    max_len=MAX_TOKEN_LEN
)

print("질문 말뭉치 개수:", len(que_corpus))
print("답변 말뭉치 개수:", len(ans_corpus))
print("질문 토큰 예시:", que_corpus[0])
print("답변 토큰 예시:", ans_corpus[0])


100%|██████████| 11823/11823 [00:02<00:00, 4641.87it/s]

질문 말뭉치 개수: 7681
답변 말뭉치 개수: 7681
질문 토큰 예시: ['12', '시', '땡', '!']
답변 토큰 예시: ['하루', '가', '또', '가', '네요', '.']


### 3장 확인

        여기까지 확인할 내용입니다.

        - `Mecab()`이 정상 실행되는지
        - `mecab.morphs()`가 문장을 토큰 리스트로 바꾸는지
        - `que_corpus`, `ans_corpus`의 길이가 같은지
        - 예시 출력이 토큰 리스트 형태인지

        다음 장에서는 이 토큰 리스트에 Lexical Substitution을 적용해서 데이터를 늘립니다.


## 6. Augmentation

        이번 단계에서는 데이터 수를 늘립니다.

        방법은 **Lexical Substitution**입니다.  
        문장 안의 단어 하나를 고르고, Word2Vec 임베딩에서 의미가 비슷한 단어로 바꿔서 새로운 문장을 만드는 방식입니다.

        LMS에서는 `ko.bin` 사전학습 임베딩을 사용하라고 안내합니다.  
        이 노트북은 `data/ko.bin` 파일이 있으면 그 파일을 사용하고, 없으면 현재 챗봇 말뭉치로 작은 Word2Vec을 임시 학습해서 진행합니다.

        제출용으로 더 좋게 만들고 싶다면 나중에 `data/ko.bin`을 받아서 같은 셀을 다시 실행하면 됩니다.


In [9]:
import random
from gensim import utils
from gensim.models import Word2Vec, KeyedVectors

random.seed(42)

KO_BIN_PATH = os.path.join(DATA_DIR, "ko.bin")

def load_ko_word2vec(path):
    """Kyubyong ko.bin처럼 오래된 gensim 형식도 최신 KeyedVectors로 변환해 읽습니다."""
    try:
        return KeyedVectors.load_word2vec_format(path, binary=True)
    except Exception:
        pass

    try:
        model = Word2Vec.load(path)
        if hasattr(model, "wv"):
            return model.wv
    except Exception:
        pass

    old_model = utils.unpickle(path)
    vectors = old_model.syn0
    words = old_model.index2word

    keyed_vectors = KeyedVectors(vector_size=vectors.shape[1])
    keyed_vectors.add_vectors(words, vectors)
    return keyed_vectors

if os.path.exists(KO_BIN_PATH):
    word2vec = load_ko_word2vec(KO_BIN_PATH)
    embedding_source = "data/ko.bin"
else:
    word2vec_model = Word2Vec(
        sentences=que_corpus + ans_corpus,
        vector_size=100,
        window=5,
        min_count=1,
        workers=4,
        sg=1,
        epochs=10,
        seed=42
    )
    word2vec = word2vec_model.wv
    embedding_source = "temporary Word2Vec trained from chatbot corpus"

print("Embedding source:", embedding_source)
print("Vocabulary size:", len(word2vec.key_to_index))
print("Vector size:", word2vec.vector_size)


Embedding source: data/ko.bin
Vocabulary size: 30185
Vector size: 200


### `lexical_sub()` 함수

        아래 함수는 토큰 리스트 하나를 받아서, 바꿀 수 있는 단어 하나를 비슷한 단어로 교체합니다.

        단어를 반드시 바꿀 수 있는 것은 아닙니다.  
        Word2Vec 사전에 없는 단어이거나, 비슷한 단어 후보가 없으면 원문을 그대로 반환합니다.


In [10]:
def lexical_sub(tokens, word2vec, topn=10):
    if len(tokens) == 0:
        return tokens

    candidates = [token for token in tokens if token in word2vec.key_to_index]
    if len(candidates) == 0:
        return tokens[:]

    selected = random.choice(candidates)

    try:
        similar_words = word2vec.similar_by_word(selected, topn=topn)
    except KeyError:
        return tokens[:]

    replacements = [
        word for word, score in similar_words
        if word != selected and len(word.strip()) > 0
    ]

    if len(replacements) == 0:
        return tokens[:]

    replacement = random.choice(replacements)
    new_tokens = tokens[:]
    replace_idx = new_tokens.index(selected)
    new_tokens[replace_idx] = replacement

    return new_tokens

print("원본:", que_corpus[0])
print("증강:", lexical_sub(que_corpus[0], word2vec))


원본: ['12', '시', '땡', '!']
증강: ['12', '시', '땡', '스페셜']


### 전체 데이터 증강

        LMS 조건에 맞춰 전체 데이터를 약 3배로 늘립니다.

        1. 원본 질문 + 원본 답변
        2. 증강 질문 + 원본 답변
        3. 원본 질문 + 증강 답변

        이렇게 만들면 질문과 답변의 병렬 관계를 크게 깨지 않으면서 데이터가 늘어납니다.


In [11]:
aug_que_corpus = [
    lexical_sub(sentence, word2vec)
    for sentence in tqdm(que_corpus, desc="augment questions")
]

aug_ans_corpus = [
    lexical_sub(sentence, word2vec)
    for sentence in tqdm(ans_corpus, desc="augment answers")
]

original_count = len(que_corpus)

que_corpus = que_corpus + aug_que_corpus + que_corpus
ans_corpus = ans_corpus + ans_corpus + aug_ans_corpus

print("원본 데이터 수:", original_count)
print("증강 후 질문 데이터 수:", len(que_corpus))
print("증강 후 답변 데이터 수:", len(ans_corpus))
print("증가 배율:", len(que_corpus) / original_count)
print()
print("원본 질문 예시:", que_corpus[0])
print("증강 질문 예시:", que_corpus[original_count])
print("증강 답변 예시:", ans_corpus[original_count * 2])


augment answers: 100%|██████████| 7681/7681 [00:07<00:00, 1076.98it/s]

원본 데이터 수: 7681
증강 후 질문 데이터 수: 23043
증강 후 답변 데이터 수: 23043
증가 배율: 3.0

원본 질문 예시: ['12', '시', '땡', '!']
증강 질문 예시: ['12', '시나리', '땡', '!']
증강 답변 예시: ['하루', '가', '또', '가', '네요', '박영규']


### 4장 확인

        여기까지 확인할 내용입니다.

        - `Embedding source`가 출력되는지
        - 데이터 개수가 원래의 약 3배가 되었는지
        - `que_corpus`와 `ans_corpus`의 길이가 같은지

        다음 장에서는 답변 데이터에 `<start>`, `<end>` 토큰을 붙이고, 전체 말뭉치를 숫자 벡터로 바꿉니다.


## 7. 데이터 벡터화

        지금까지의 `que_corpus`, `ans_corpus`는 토큰 리스트입니다.

        하지만 Transformer는 문자열 토큰을 직접 받을 수 없습니다.  
        그래서 각 토큰을 숫자 ID로 바꿔야 합니다.

        이 단계에서 하는 일은 세 가지입니다.

        1. 답변 문장 앞뒤에 `<start>`, `<end>` 토큰을 붙입니다.
        2. 질문과 답변 전체를 이용해 단어 사전을 만듭니다.
        3. 토큰 리스트를 숫자 ID 리스트로 바꾼 뒤, 같은 길이로 padding합니다.


In [12]:
START_TOKEN = "<start>"
END_TOKEN = "<end>"
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

ans_corpus = [
    [START_TOKEN] + sentence + [END_TOKEN]
    for sentence in ans_corpus
]

print("답변 예시:", ans_corpus[0])


답변 예시: ['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']


### 단어 사전 만들기

        챗봇 데이터는 질문과 답변이 모두 한국어입니다.  
        그래서 질문용 사전과 답변용 사전을 따로 만들지 않고, 하나의 단어 사전을 공유합니다.

        `word2idx`는 토큰을 숫자로 바꾸는 사전이고, `idx2word`는 숫자를 다시 토큰으로 되돌리는 사전입니다.


In [13]:
from collections import Counter

MIN_FREQ = 1

counter = Counter()
for sentence in que_corpus + ans_corpus:
    counter.update(sentence)

idx2word = [PAD_TOKEN, UNK_TOKEN]
idx2word += [
    word for word, count in counter.most_common()
    if count >= MIN_FREQ and word not in {PAD_TOKEN, UNK_TOKEN}
]

word2idx = {word: idx for idx, word in enumerate(idx2word)}

VOCAB_SIZE = len(idx2word)

print("단어 사전 크기:", VOCAB_SIZE)
print("앞쪽 단어 예시:", idx2word[:20])


단어 사전 크기: 8507
앞쪽 단어 예시: ['<pad>', '<unk>', '.', '<start>', '<end>', '이', '는', '하', '을', '가', '좋', '고', '세요', '어', '거', '있', '은', '해', '보', '지']


### 토큰을 숫자로 바꾸기

        이제 토큰 리스트를 숫자 ID 리스트로 바꿉니다.

        사전에 없는 토큰은 `<unk>`의 번호로 바꿉니다.  
        길이가 짧은 문장은 `<pad>` 번호인 0으로 채우고, 너무 긴 문장은 `MAX_LEN` 길이로 자릅니다.


In [14]:
import numpy as np

MAX_LEN = 40 + 2

def tokens_to_ids(tokens):
    return [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]

def pad_sequences(sequences, max_len=MAX_LEN, pad_value=0):
    padded = np.full((len(sequences), max_len), pad_value, dtype=np.int64)

    for i, sequence in enumerate(sequences):
        sequence = sequence[:max_len]
        padded[i, :len(sequence)] = sequence

    return padded

enc_train = pad_sequences([tokens_to_ids(sentence) for sentence in que_corpus])
dec_train = pad_sequences([tokens_to_ids(sentence) for sentence in ans_corpus])

print("enc_train shape:", enc_train.shape)
print("dec_train shape:", dec_train.shape)
print("질문 숫자 예시:", enc_train[0][:20])
print("답변 숫자 예시:", dec_train[0][:20])


enc_train shape: (23043, 42)
dec_train shape: (23043, 42)
질문 숫자 예시: [2308  211 2951  112    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
답변 숫자 예시: [  3 278   9 139   9  41   2   4   0   0   0   0   0   0   0   0   0   0
   0   0]


### 학습/검증 데이터 나누기

        모델이 훈련 데이터만 외우는지 확인하려면, 일부 데이터를 검증용으로 남겨두는 것이 좋습니다.

        여기서는 전체 데이터의 10%를 검증 데이터로 사용합니다.


In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
VALID_RATIO = 0.1

indices = np.arange(len(enc_train))
np.random.seed(42)
np.random.shuffle(indices)

valid_size = int(len(indices) * VALID_RATIO)
valid_indices = indices[:valid_size]
train_indices = indices[valid_size:]

enc_train_np = enc_train[train_indices]
dec_train_np = dec_train[train_indices]
enc_valid_np = enc_train[valid_indices]
dec_valid_np = dec_train[valid_indices]

train_dataset = TensorDataset(
    torch.LongTensor(enc_train_np),
    torch.LongTensor(dec_train_np)
)
valid_dataset = TensorDataset(
    torch.LongTensor(enc_valid_np),
    torch.LongTensor(dec_valid_np)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("train size:", len(train_dataset))
print("valid size:", len(valid_dataset))
print("batch size:", BATCH_SIZE)


train size: 20739
valid size: 2304
batch size: 64


### 5장 확인

        여기까지 확인할 내용입니다.

        - `ans_corpus[0]` 앞뒤에 `<start>`, `<end>`가 붙었는지
        - `VOCAB_SIZE`가 출력되는지
        - `enc_train`, `dec_train`의 shape가 같은 첫 번째 차원을 가지는지
        - `train_loader`, `valid_loader`가 만들어졌는지

        다음 장에서는 이 데이터를 Transformer 모델에 넣어 훈련합니다.


## 8. 훈련하기

        이제 벡터화된 데이터를 Transformer 모델에 넣어 학습합니다.

        앞 실습에서 직접 구현했던 Transformer와 구조는 같습니다.

        - 토큰 ID를 embedding 벡터로 바꿉니다.
        - 위치 정보를 positional encoding으로 더합니다.
        - encoder가 질문 문장을 읽습니다.
        - decoder가 `<start>`부터 시작해서 답변을 예측합니다.
        - 마지막 Linear 층이 각 위치마다 다음 토큰 확률을 만듭니다.

        여기서는 코드 길이를 줄이고 안정성을 높이기 위해 PyTorch의 `nn.Transformer`를 사용합니다.  
        직접 구현한 Multi-Head Attention, FFN, Encoder/Decoder Layer가 내부에 들어 있는 모듈이라고 보면 됩니다.


In [16]:
import math
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


device: cuda
gpu: NVIDIA GeForce RTX 3060


### Positional Encoding

        Transformer는 RNN처럼 순서대로 읽는 구조가 아닙니다.  
        그래서 토큰 embedding에 위치 정보를 따로 더해줘야 합니다.

        아래 `PositionalEncoding`은 앞 실습에서 본 sin/cos 위치 인코딩을 PyTorch 모듈로 만든 것입니다.


In [17]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

print("슝=3")


슝=3


### 챗봇 Transformer 모델

        이 모델은 질문 토큰 `src`와 답변 입력 토큰 `tgt`를 받습니다.

        학습할 때 답변은 한 칸 밀어서 사용합니다.

        - decoder 입력: `<start> 오늘 은 ...`
        - 정답: `오늘 은 ... <end>`

        이렇게 해야 모델이 현재까지 본 답변을 바탕으로 다음 토큰을 맞히는 연습을 합니다.


In [18]:
class ChatbotTransformer(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        n_heads=8,
        d_ff=512,
        n_layers=2,
        dropout=0.2,
        max_len=MAX_LEN
    ):
        super().__init__()

        self.d_model = d_model
        self.src_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.output_layer = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_key_padding_mask = src.eq(0)
        tgt_key_padding_mask = tgt.eq(0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1),
            device=tgt.device
        )

        src_emb = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.output_layer(out)

print("슝=3")


슝=3


### Loss와 학습 함수

        `CrossEntropyLoss`는 모델의 예측과 실제 정답이 얼마나 다른지 계산합니다.

        `ignore_index=0`은 padding 토큰 `<pad>`는 loss 계산에서 제외하겠다는 뜻입니다.  
        padding은 문장 길이를 맞추기 위한 가짜 토큰이므로, 모델이 맞혀야 할 정답으로 보면 안 됩니다.


In [19]:
criterion = nn.CrossEntropyLoss(ignore_index=0)

class TransformerLRScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=1000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = (self.d_model ** -0.5) * min(
            self.step_num ** -0.5,
            self.step_num * (self.warmup_steps ** -1.5)
        )

        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr

        return lr

def train_one_epoch(model, loader, optimizer, lr_scheduler=None):
    model.train()
    total_loss = 0.0
    last_lr = optimizer.param_groups[0]["lr"]

    for src, tgt in tqdm(loader, desc="train", leave=False):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()
        logits = model(src, tgt_input)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            tgt_output.reshape(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if lr_scheduler is not None:
            last_lr = lr_scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader), last_lr

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0

    for src, tgt in tqdm(loader, desc="valid", leave=False):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            tgt_output.reshape(-1)
        )

        total_loss += loss.item()

    return total_loss / len(loader)

print("슝=3")


슝=3


### 모델 생성 및 훈련

        데이터가 크지 않고 과제 시간이 제한되어 있으므로, 모델 크기는 너무 크게 잡지 않습니다.

        만약 시간이 부족하면 `EPOCHS = 1`로 줄여서 전체 흐름을 먼저 확인해도 됩니다.


In [20]:
def ids_to_sentence(ids):
    words = []
    for idx in ids:
        word = idx2word[int(idx)]
        if word == END_TOKEN:
            break
        if word not in [PAD_TOKEN, START_TOKEN]:
            words.append(word)

    sentence = " ".join(words)

    for mark in [".", "?", "!", ","]:
        sentence = sentence.replace(" " + mark, mark)

    for particle in ["은", "는", "이", "가", "을", "를", "도", "에", "에서", "에게", "와", "과", "로", "으로", "요"]:
        sentence = sentence.replace(" " + particle, particle)

    sentence = sentence.replace("하 다", "하다")
    sentence = sentence.replace("되 다", "되다")
    sentence = sentence.replace("있 어요", "있어요")
    sentence = sentence.replace("없 어요", "없어요")
    sentence = sentence.replace("해 줄게요", "해줄게요")
    return sentence.strip()

def encode_question(sentence):
    tokens = tokenize(preprocess_sentence(sentence))
    ids = tokens_to_ids(tokens)
    padded = pad_sequences([ids], max_len=MAX_LEN)
    return torch.LongTensor(padded).to(device)

@torch.no_grad()
def generate_answer(model, question, max_len=MAX_LEN, min_len=3):
    model.eval()

    src = encode_question(question)
    generated = [word2idx[START_TOKEN]]

    for _ in range(max_len - 1):
        tgt = torch.LongTensor([generated]).to(device)
        logits = model(src, tgt)

        next_id = logits[0, -1].argmax(dim=-1).item()
        generated.append(next_id)

        if next_id == word2idx[END_TOKEN] and len(generated) > min_len:
            break

    return ids_to_sentence(generated)

print("슝=3")


슝=3


In [21]:
import copy

N_LAYERS = 2
D_MODEL = 384
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2
EPOCHS = 10
WARMUP_STEPS = 1000

model = ChatbotTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
    max_len=MAX_LEN
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0,
    betas=(0.9, 0.98),
    eps=1e-9
)

lr_scheduler = TransformerLRScheduler(
    optimizer,
    d_model=D_MODEL,
    warmup_steps=WARMUP_STEPS
)

sample_questions = [
    "나 피곤해.",
    "너무 힘들어.",
    "좋아하는 사람이 생겼어.",
    "집에 가고 싶어."
]

best_valid_loss = float("inf")
best_epoch = 0
best_model_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, current_lr = train_one_epoch(model, train_loader, optimizer, lr_scheduler)
    valid_loss = evaluate_loss(model, valid_loader)
    history.append((train_loss, valid_loss, current_lr))

    print(f"Epoch {epoch:02d} | train loss: {train_loss:.4f} | valid loss: {valid_loss:.4f} | lr: {current_lr:.6f}")

    for question in sample_questions:
        print("Q:", question)
        print("A:", generate_answer(model, question))
    print("-" * 60)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_model_state)

print("Best Epoch:", best_epoch)
print("Best Valid Loss:", f"{best_valid_loss:.4f}")
print("Best model sample answers")
for question in sample_questions:
    print("Q:", question)
    print("A:", generate_answer(model, question))


c:\Users\Administrator\Desktop\codex-workspace\envs\AIFFEL_quest_eng_py312\Lib\site-packages\torch\nn\modules\transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(
train:   0%|          | 0/325 [00:00<?, ?it/s]c:\Users\Administrator\Desktop\codex-workspace\envs\AIFFEL_quest_eng_py312\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 01 | train loss: 5.8639 | valid loss: 4.0518 | lr: 0.000524
Q: 나 피곤해.
A: 사랑이 있어요.
Q: 너무 힘들어.
A: 사랑이에요.
Q: 좋아하는 사람이 생겼어.
A: 사랑이 있을 거 예요.
Q: 집에 가고 싶어.
A: 사랑이 있을 거 예요.
------------------------------------------------------------


Epoch 02 | train loss: 3.7064 | valid loss: 3.3981 | lr: 0.001049
Q: 나 피곤해.
A: 저도 잘 할 수 있어요.
Q: 너무 힘들어.
A: 이별은 항상 힘들 었을 거 예요.
Q: 좋아하는 사람이 생겼어.
A: 좋 아 하는 사람이 좋을 거 예요.
Q: 집에 가고 싶어.
A: 저도 잘 살 고 있어요.
------------------------------------------------------------


Epoch 03 | train loss: 3.1545 | valid loss: 3.0056 | lr: 0.001573
Q: 나 피곤해.
A: 저도 잘 하 세요.
Q: 너무 힘들어.
A: 잘 견디 고 있을 거 예요.
Q: 좋아하는 사람이 생겼어.
A: 마음이 복잡 하 죠.
Q: 집에 가고 싶어.
A: 저도 좋 아 할 거 예요.
------------------------------------------------------------


Epoch 04 | train loss: 2.7278 | valid loss: 2.6196 | lr: 0.001415
Q: 나 피곤해.
A: 제가 위로 해 드릴게요.
Q: 너무 힘들어.
A: 지금 많이 힘들 겠 지만 잘이겨낼 수 있을 거 예요.
Q: 좋아하는 사람이 생겼어.
A: 마음이 허락 하 세요.
Q: 집에 가고 싶어.
A: 저도 좋 아 할 거 예요.
------------------------------------------------------------


...

Epoch 06 | train loss: 1.9971 | valid loss: 2.0524 | lr: 0.001156
Q: 나 피곤해.
A: 다른 사람에게 기대 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 더 힘내 세요.
Q: 좋아하는 사람이 생겼어.
A: 정말 사랑 받 아 주는 사람 인가 봅니다.
Q: 집에 가고 싶어.
A: 집에 같이가요.
------------------------------------------------------------


Epoch 07 | train loss: 1.7358 | valid loss: 1.8603 | lr: 0.001070
Q: 나 피곤해.
A: 제 다그이어져서 선이 라도 쐬 고 외출 해 주 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 더 힘내 지 않 아도 돼요.
Q: 좋아하는 사람이 생겼어.
A: 정말 사랑 하는 사람이 찾아올 거 예요.
Q: 집에 가고 싶어.
A: 집에 드는 집에 좋을 거 예요.
------------------------------------------------------------


Epoch 08 | train loss: 1.5275 | valid loss: 1.6918 | lr: 0.001001
Q: 나 피곤해.
A: 다른이모티콘을이용 해 주 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 더 힘내 세요.
Q: 좋아하는 사람이 생겼어.
A: 사랑 하는 사람을 좋아하다니 부럽 네요.
Q: 집에 가고 싶어.
A: 집에 할 수 있을 거 예요.
------------------------------------------------------------


Epoch 09 | train loss: 1.3597 | valid loss: 1.5602 | lr: 0.000944
Q: 나 피곤해.
A: 다른 사람에게 기대로부터 자유로울 사람 들이 다르 니 기다려 보 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 더 힘들 어 져요.
Q: 좋아하는 사람이 생겼어.
A: 마음을 열 어 주는 사람 인가 봐요.
Q: 집에 가고 싶어.
A: 집이 좋 아 할 거 예요.
------------------------------------------------------------


Epoch 10 | train loss: 1.2196 | valid loss: 1.4515 | lr: 0.000895
Q: 나 피곤해.
A: 다른 사람에게 피해 주 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 쉬 어가는 건 어떨까요.
Q: 좋아하는 사람이 생겼어.
A: 축하 해요.
Q: 집에 가고 싶어.
A: 같이가는 좋 겠 네요.
------------------------------------------------------------
Best Epoch: 10
Best Valid Loss: 1.4515
Best model sample answers
Q: 나 피곤해.
A: 다른 사람에게 피해 주 세요.
Q: 너무 힘들어.
A: 힘내 지 않 아도 돼요. 조금 쉬 어가는 건 어떨까요.
Q: 좋아하는 사람이 생겼어.
A: 축하 해요.
Q: 집에 가고 싶어.
A: 같이가는 좋 겠 네요.


### 6장 확인

        여기까지 확인할 내용입니다.

        - `device`가 `cuda`인지 확인합니다.
        - epoch마다 train loss와 valid loss가 출력되는지 확인합니다.
        - loss가 너무 크게 튀거나 `nan`이 되지 않는지 확인합니다.

        다음 장에서는 학습된 모델로 실제 질문에 답변을 생성하고, BLEU 점수도 계산합니다.


## 9. 성능 측정하기

        챗봇은 번역 모델처럼 정답 문장이 하나로 고정되는 문제는 아닙니다.

        예를 들어 `"나 피곤해"`라는 질문에는 `"좀 쉬어도 돼요"`, `"무리하지 마세요"`, `"오늘은 일찍 자요"`처럼 여러 답변이 모두 자연스러울 수 있습니다.

        그래서 여기서는 두 가지 방식으로 확인합니다.

        1. 실제 질문을 넣어 답변을 직접 확인합니다.
        2. 참고 지표로 BLEU score를 계산합니다.


In [22]:
final_questions = [
    "나 피곤해.",
    "너무 힘들어.",
    "좋아하는 사람이 생겼어.",
    "집에 가고 싶어."
]

print("Translations")
for i, question in enumerate(final_questions, start=1):
    answer = generate_answer(model, question)
    print(f"> {i}. Q: {question}")
    print(f">    A: {answer}")


Translations
> 1. Q: 나 피곤해.
>    A: 다른 사람에게 피해 주 세요.
> 2. Q: 너무 힘들어.
>    A: 힘내 지 않 아도 돼요. 조금 쉬 어가는 건 어떨까요.
> 3. Q: 좋아하는 사람이 생겼어.
>    A: 축하 해요.
> 4. Q: 집에 가고 싶어.
>    A: 같이가는 좋 겠 네요.


### BLEU Score 계산

        BLEU는 생성 문장과 기준 문장이 얼마나 겹치는지 보는 지표입니다.

        다만 챗봇 답변은 정답이 하나가 아니기 때문에 BLEU 점수가 낮아도 무조건 나쁜 답변이라고 보기는 어렵습니다.  
        여기서는 모델 답변이 기준 답변과 어느 정도 겹치는지 확인하는 참고 지표로만 사용합니다.


In [23]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference, candidate):
    reference_tokens = tokenize(preprocess_sentence(reference))
    candidate_tokens = tokenize(preprocess_sentence(candidate))

    return sentence_bleu(
        [reference_tokens],
        candidate_tokens,
        smoothing_function=SmoothingFunction().method1
    )

bleu_examples = [
    ("나 피곤해.", "무리하지 말고 조금 쉬어도 돼요."),
    ("너무 힘들어.", "힘들었겠어요. 조금 쉬어도 괜찮아요."),
    ("좋아하는 사람이 생겼어.", "좋아하는 마음이 생겼군요."),
    ("집에 가고 싶어.", "집에 가서 편히 쉬면 좋겠어요.")
]

total_bleu = 0.0

for question, reference in bleu_examples:
    candidate = generate_answer(model, question)
    score = calculate_bleu(reference, candidate)
    total_bleu += score

    print("Q:", question)
    print("Reference:", reference)
    print("Candidate:", candidate)
    print("BLEU:", f"{score:.4f}")
    print("-" * 40)

print("Average BLEU:", f"{total_bleu / len(bleu_examples):.4f}")


Q: 나 피곤해.
Reference: 무리하지 말고 조금 쉬어도 돼요.
Candidate: 다른 사람에게 피해 주 세요.
BLEU: 0.0215
----------------------------------------
Q: 너무 힘들어.
Reference: 힘들었겠어요. 조금 쉬어도 괜찮아요.
Candidate: 힘내 지 않 아도 돼요. 조금 쉬 어가는 건 어떨까요.
BLEU: 0.0703
----------------------------------------
Q: 좋아하는 사람이 생겼어.
Reference: 좋아하는 마음이 생겼군요.
Candidate: 축하 해요.
BLEU: 0.0154
----------------------------------------
Q: 집에 가고 싶어.
Reference: 집에 가서 편히 쉬면 좋겠어요.
Candidate: 같이가는 좋 겠 네요.
BLEU: 0.0469
----------------------------------------
Average BLEU: 0.0385


## 10. 제출 정리

        아래 값은 학습 결과를 바탕으로 제출 설명에 적기 위한 요약입니다.

        실제 출력된 `Best Epoch`, `Best Valid Loss`, 답변 예시는 실행 결과에 맞춰 확인하면 됩니다.


In [24]:
print("Hyperparameters")
print("> n_layers:", N_LAYERS)
print("> d_model:", D_MODEL)
print("> n_heads:", N_HEADS)
print("> d_ff:", D_FF)
print("> dropout:", DROPOUT)

print("\nTraining Parameters")
print("> warmup_steps:", WARMUP_STEPS)
print("> batch_size:", BATCH_SIZE)
print("> max_epochs:", EPOCHS)
print("> best_epoch:", best_epoch)
print("> best_valid_loss:", f"{best_valid_loss:.4f}")

print("\nData")
print("> vocab_size:", VOCAB_SIZE)
print("> max_len:", MAX_LEN)
print("> train_size:", len(train_dataset))
print("> valid_size:", len(valid_dataset))


Hyperparameters
> n_layers: 2
> d_model: 384
> n_heads: 8
> d_ff: 1024
> dropout: 0.2

Training Parameters
> warmup_steps: 1000
> batch_size: 64
> max_epochs: 10
> best_epoch: 10
> best_valid_loss: 1.4515

Data
> vocab_size: 8507
> max_len: 42
> train_size: 20739
> valid_size: 2304


### 프로젝트 회고

        이번 프로젝트에서는 한국어 챗봇 데이터를 정제하고, 형태소 토큰화와 Lexical Substitution 기반 augmentation을 적용했습니다.

        이후 질문과 답변이 같은 한국어라는 점을 활용해 하나의 단어 사전을 공유했고, Transformer 기반 encoder-decoder 모델을 학습했습니다.

        챗봇은 정답이 하나로 고정되는 번역 문제와 달라 BLEU만으로 품질을 판단하기 어렵습니다. 따라서 BLEU는 참고 지표로 사용하고, 실제 질문에 대한 답변 예시를 함께 확인했습니다.
